# Qwen System-One v1.1.0 — Fine-tuning (receita Decider 0.8B + multilíngue)

**Base:** `Qwen/Qwen3.5-0.8B-Base` · **Receita:** 1 época, LR 1e-5, warmup 150, readout letter-logits + softmax  
**Diferencial:** prompt no formato do nosso runtime (template primeiro) + multilíngue PT/ES/DE/ZH  
**Licença:** Apache 2.0 (mixture de dados públicos — sem contaminação do Decision Index/JevBench)

Ordem: setup → dados (EN + multilíngue) → treino → calibração T → salva no Drive.

## 0. O que você precisa

| Item | Detalhe |
|---|---|
| **GPU** | A100 40GB (Colab Pro) p/ full-finetune; T4/L4 grátis funciona com QLoRA (célula alternativa) |
| **Tempo** | ~4.5h (GH200 ref) — no Colab A100, uma tarde |
| **Drive** | TUDO cacheado: base Qwen + tasks.pkl + parquets + script — Colab NÃO baixa nada (zero download) |
| **Token HF** | NÃO precisa (tudo já está no Drive) |

In [ ]:
# Célula 1 — setup: Drive + GPU check + instala decider do git
# (idempotente: pode rodar quantas vezes quiser)
# TUDO vem do Drive (base, tasks.pkl, parquets) — ZERO download no Colab.
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print('Drive:', e, '— seguindo com disco local (treino será perdido se a sessão cair!)')

import os
GROOT = '/content/drive/MyDrive/qwen-system-one'
USE_DRIVE = os.path.isdir(GROOT)
OUT = (GROOT + '/runs/decider08_full' if USE_DRIVE else '/content/runs/decider08_full')
!mkdir -p $OUT /content/work && cd /content/work

!nvidia-smi --query-gpu=name,memory.total --format=csv | head -n 3
!test -d decider || (tar -xzf /content/drive/MyDrive/qwen-system-one/repo/decider-repo.tar.gz && mv .decider-mirror decider) 2>&1 | tail -n 1
!pip install -q ./decider[train] 2>&1 | tail -n 2
print('OUT =', OUT)
# Verificação offline: base + tasks.pkl + parquets já no Drive?
import os as _os
_chk = {
    'base/model.safetensors': GROOT + '/models/base/model.safetensors',
    'data/tasks.pkl': GROOT + '/data/tasks.pkl',
    'data/train (5 parquets)': GROOT + '/data/train',
}
_allok = True
for _k, _p in _chk.items():
    _ok = _os.path.exists(_p)
    _allok = _allok and _ok
    print(('OK  ' if _ok else 'FALTA') + ' ' + _k)
print('OFFLINE PRONTO' if _allok else 'FALTA ARQUIVO NO DRIVE — veja acima')# VRAM check: full-finetune 0.8B precisa ~25-30GB; QLoRA ~12-15GB
!nvidia-smi --query-gpu=memory.free --format=csv,noheader | head -n 1


## 1. Dados: mixture EN (receita Decider) + multilíngue

A mixture base vem do `decider.data` (~95 datasets públicos, `teacher_data/` já no repo). 
O bloco multilíngue adiciona intent/triagem em **PT/ES/DE/ZH** no formato typed-decision — é o diferencial contra Decider/JPT (English-only) e contra o Laya (colapsa fora do inglês).

⚠️ **Regra de ouro:** NADA do Decision Index frozen suite, NADA do JevBench (público ou sealed), NADA do EnvBench held-out. O index é a prova cega.

In [ ]:
# Célula 2 — monta mixture EN + multilíngue (TUDO do Drive, ZERO download)
# Drive/qwen-system-one contém: models/base/, data/tasks.pkl, data/train/*.parquet
import pickle, json, hashlib, os, shutil, subprocess
from pathlib import Path

GROOT = '/content/drive/MyDrive/qwen-system-one'
DATA = GROOT + '/data/train'
print('DATA =', DATA)
for f in ['minds14-en-US.parquet', 'minds14-de-DE.parquet', 'minds14-es-ES.parquet', 'minds14-pt-PT.parquet', 'minds14-zh-CN.parquet']:
    p = os.path.join(DATA, f)
    print(('OK  ' if os.path.exists(p) else 'FALTA') + ' ' + f)

BASE = 'data/tasks.pkl'
TASKS_DRIVE = GROOT + '/data/tasks.pkl'
!test -d /content/work/decider || (cd /content/work && tar -xzf /content/drive/MyDrive/qwen-system-one/repo/decider-repo.tar.gz && mv .decider-mirror decider) 2>&1 | tail -n 1
# tasks.pkl vem do Drive (sem download). Só gera se faltar nos DOIS lugares.
!if [ -f "{TASKS_DRIVE}" ]; then echo 'tasks.pkl no Drive — copiando (sem download)'; mkdir -p /content/work/decider/data && cp "{TASKS_DRIVE}" /content/work/decider/data/tasks.pkl && ls -lh /content/work/decider/data/tasks.pkl; else echo 'tasks.pkl NÃO está no Drive — gerando (demora ~40min, 1x só)'; fi
!test -f /content/work/decider/data/tasks.pkl || (cd /content/work/decider && python -u -m decider.data.core --out {BASE} 2>&1 | tee /tmp/core.log | grep -E "OK |SKIP |tasks:|ERROR |Traceback" | tail -n 30; echo "--- fim (log completo em /tmp/core.log) ---"; tail -n 3 /tmp/core.log)
# Salva o tasks.pkl no Drive p/ reaproveitar nas próximas sessões
!if [ -f /content/work/decider/data/tasks.pkl ] && [ ! -f "{TASKS_DRIVE}" ]; then cp /content/work/decider/data/tasks.pkl "{TASKS_DRIVE}" && echo 'tasks.pkl salvo no Drive'; fi; ls -lh /content/work/decider/data/tasks.pkl "{TASKS_DRIVE}" 2>&1

# Bloco multilíngue: gera do parquet local (SEM download, SEM datasets) via script do repo
# Script TAMBÉM no Drive (100% offline): sempre sobrescreve (garante versão nova)
!cp /content/drive/MyDrive/qwen-system-one/repo/tools/make_multi_typed.py /content/work/ 2>/dev/null || curl -sL -o /content/work/make_multi_typed.py https://raw.githubusercontent.com/italoalmeida0/qwen-system-one/main/tools/make_multi_typed.py; grep -c by_idx /content/work/make_multi_typed.py
!python /content/work/make_multi_typed.py --src $DATA --out /content/work/multi_typed.jsonl

# Bloco multilíngue gerado acima via make_multi_typed.py (MINDS-14, 5 idiomas).
# Verificação rápida do JSONL gerado:
!wc -l /content/work/multi_typed.jsonl && head -c 400 /content/work/multi_typed.jsonl


## 2. Overlap check (8-gram) — anti-contaminação

Antes de treinar: garante que NENHUM exemplo de treino aparece nos benchmarks (Decision Index frozen, JevBench, EnvBench held-out). O próprio index invalida quem treina na prova.

In [ ]:
# Célula 3 — overlap check (multi vs benchmark + duplicatas internas)
def grams8(text):
    toks = text.lower().split()
    return {' '.join(toks[i:i+8]) for i in range(max(0, len(toks)-7))}

import json
FORBIDDEN = set()
try:
    from datasets import load_dataset
    di = load_dataset('multimodalart/jev-decision-index', split='train')
    for r in di:
        t = r.get('text') or r.get('prompt') or r.get('question') or ''
        FORBIDDEN |= grams8(str(t))
    print(f'benchmark rows: {len(di)}, 8-grams: {len(FORBIDDEN)}')
except Exception as e:
    print('benchmark inacessível:', str(e)[:150])

bad, seen, dup = 0, set(), 0
for line in open('/content/work/multi_typed.jsonl'):
    ex = json.loads(line)
    g = grams8(ex['prompt'])
    if FORBIDDEN and (g & FORBIDDEN): bad += 1
    h = hash(ex['prompt'])
    if h in seen: dup += 1
    seen.add(h)
print(f'vs benchmark: {bad} (=0) | duplicatas internas: {dup}')


## 3. Treino (receita Decider, base 0.8B)

`train.sh full` traduzido: 1 época, LR 1e-5, warmup 150, `max_tokens 16384`, `max_options 255`, `schema_first_prob 0.5` (metade schema-first = nossa ordem de prompt), `none_prob 0.1`. Readout: letter-logits no answer slot + softmax.

In [ ]:
# Célula 4 — treino full (A100 80GB). Para T4/L4 grátis, ver célula 4b (QLoRA).
# Base 100% do Drive (ZERO download): copia p/ disco local 1x
!mkdir -p /content/work/base && test -f /content/work/base/model.safetensors || cp /content/drive/MyDrive/qwen-system-one/models/base/* /content/work/base/ && ls -lh /content/work/base/
!cd /content/work/decider && python -u -m decider.data.mixture --base data/tasks.pkl --mode full --out data/mixture_full.pkl --probes data/probes.pkl 2>&1 | tee /tmp/mixture.log | tail -n 5

# INJETA o bloco multilíngue na mixture (formato Example/Q interno — sem isso o treino IGNORA o JSONL!)
!cp /content/drive/MyDrive/qwen-system-one/repo/tools/inject_multi.py /content/work/decider/ 2>/dev/null || curl -sL -o /content/work/decider/inject_multi.py https://raw.githubusercontent.com/italoalmeida0/qwen-system-one/main/tools/inject_multi.py; ls -la /content/work/decider/inject_multi.py
!cd /content/work/decider && python inject_multi.py --mixture data/mixture_full.pkl --jsonl /content/work/multi_typed.jsonl --out data/mixture_full.pkl --task minds14_multi

!cd /content/work/decider && python -m decider.train --model /content/work/base --data data/mixture_full.pkl --out "$OUT" --epochs 1 --lr 1e-5 --warmup 150 --max_tokens 16384 --accum 2 --max_options 255 --max_ctx 16384 --none_prob 0.1 --schema_first_prob 0.5 --eval_every 100000 --eval_limit 150 2>&1 | tee "$OUT/../train_full.log"

In [ ]:
# Célula 4b — ALTERNATIVA QLoRA (T4/L4 grátis, 15GB VRAM). Mais lenta, mesmo formato.
# !pip install -q bitsandbytes peft 2>&1 | tail -n 1
# Mesma mixture, com quant 4-bit + LoRA r=16 em todas as projeções (receita JPT):
# !cd /content/work/decider && python -m decider.train --model /content/work/base --data data/mixture_full.pkl --out "$OUT" --epochs 1 --lr 5e-5 --warmup 150 --qlora --lora_r 16 --max_options 255 --schema_first_prob 0.5 2>&1 | tee "$OUT/../train_qlora.log"

## 4. Calibração da temperatura + eval + export

Calibra T em split held-out (JPT: 1.140; Decider-0.8B: 1.03 — o valor final sai daqui). Depois: merge → ONNX → quant Q4 nos 2 arquivos do runtime.

In [ ]:
# Célula 5 — calibra T, avalia, salva
!cd /content/work/decider && python -m decider.calibrate --model "$OUT/model" 2>&1 | tail -n 5
!cd /content/work/decider && bash scripts/evaluate.sh "$OUT/model" 2>&1 | tail -n 10
print('Modelo em:', OUT + '/model')
print('Próximo: merge + export ONNX Q4 -> embed_tokens_q4.onnx + decoder_model_merged_q4.onnx (Fase C)')